In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import glob
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# --------------------
# 1. Load all HEV + PHEV CSV files
# --------------------
folder_path = '/content/drive/MyDrive/Data/VED-dataset/VED_HEV_PHEV'
all_files = glob.glob(os.path.join(folder_path, '*.csv')) + glob.glob(os.path.join(folder_path, '*.CSV'))

print(f"Number of files found: {len(all_files)}")

df_list = []
for file in all_files:
    try:
        df_temp = pd.read_csv(file)
        if not df_temp.empty:
            df_list.append(df_temp)
    except Exception as e:
        print(f"Skipping file {file} due to error: {e}")

# Combine all files into one DataFrame
if len(df_list) == 0:
    raise ValueError("No CSV files were loaded. Check the folder path or file extensions.")

df = pd.concat(df_list, ignore_index=True)
print(f"Total rows combined: {len(df):,}")

Number of files found: 54
Total rows combined: 8,830,937


In [ ]:
# --------------------
# 2. Select features and targets
# --------------------
target_cols = ['Fuel Rate[L/hr]', 'HV Battery SOC[%]']
for col in target_cols:
    if col not in df.columns:
        raise ValueError(f"Column {col} not found in data. Available columns: {df.columns.tolist()}")

# Drop rows with missing target values
df = df.dropna(subset=target_cols)

# Choose features (drop target columns and non-numeric columns)
features = df.drop(columns=target_cols)
features = features.select_dtypes(include=[np.number])  # Only numeric columns

# Drop columns with all NaNs or zero variance
features = features.dropna(axis=1, how='all')
features = features.loc[:, features.var() != 0]

# Drop any remaining rows with NaN in features
df = df.loc[features.index]
df = df.dropna(subset=features.columns)

X = features.values
y = df[target_cols].values

print(f"Feature shape: {X.shape}, Target shape: {y.shape}")

Feature shape: (894275, 13), Target shape: (402050, 2)


In [ ]:
# --------------------
# 3. Split data: 10% validation, then 80/20 train/test
# --------------------
'''X_temp, X_val, y_temp, y_val = train_test_split(X, y, test_size=0.10, shuffle=True, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_temp, y_temp, test_size=0.20, shuffle=True, random_state=42)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}, Validation size: {len(X_val)}")'''

# CRITICAL STEP: Sort the DataFrame chronologically before splitting
# The 'DayNum' and 'Timestamp(ms)' columns are ideal for this.
df = df.sort_values(by=['DayNum', 'Timestamp(ms)']).reset_index(drop=True)

# Define features and targets again after sorting
# This is necessary because the row order has changed.
features = df.drop(columns=target_cols)
features = features.select_dtypes(include=[np.number])
features = features.dropna(axis=1, how='all')
features = features.loc[:, features.var() != 0]

df = df.loc[features.index]
df = df.dropna(subset=features.columns)

X = features.values
y = df[target_cols].values

# Perform chronological split using slicing
# 10% for final validation set
validation_split_point = int(len(X) * 0.90)
X_temp = X[:validation_split_point]
X_val = X[validation_split_point:]
y_temp = y[:validation_split_point]
y_val = y[validation_split_point:]

# 80/20 split on the remaining data for training and testing
train_split_point = int(len(X_temp) * 0.80)
X_train = X_temp[:train_split_point]
X_test = X_temp[train_split_point:]
y_train = y_temp[:train_split_point]
y_test = y_temp[train_split_point:]

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}, Validation size: {len(X_val)}")

Train size: 289476, Test size: 72369, Validation size: 40205


In [ ]:
# --------------------
# 4. Normalize features
# --------------------
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)
X_val = scaler_X.transform(X_val)

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train)
y_test = scaler_y.transform(y_test)
y_val = scaler_y.transform(y_val)

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

In [ ]:
# --------------------
# 5. Define PyTorch Neural Network
# --------------------
class HEVModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(HEVModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )
    def forward(self, x):
        return self.net(x)

model = HEVModel(input_dim=X_train.shape[1], output_dim=y_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# --------------------
# 6. Training Loop
# --------------------
epochs = 50
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

    # Validation loss
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_t)
        val_loss = criterion(val_outputs, y_val_t)

    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}")

Epoch [1/50], Train Loss: 0.2773, Val Loss: 0.3249
Epoch [2/50], Train Loss: 0.2568, Val Loss: 0.3083
Epoch [3/50], Train Loss: 0.2387, Val Loss: 0.2890
Epoch [4/50], Train Loss: 0.2230, Val Loss: 0.2678
Epoch [5/50], Train Loss: 0.2094, Val Loss: 0.2458
Epoch [6/50], Train Loss: 0.1976, Val Loss: 0.2243
Epoch [7/50], Train Loss: 0.1874, Val Loss: 0.2048
Epoch [8/50], Train Loss: 0.1783, Val Loss: 0.1881
Epoch [9/50], Train Loss: 0.1701, Val Loss: 0.1748
Epoch [10/50], Train Loss: 0.1626, Val Loss: 0.1650
Epoch [11/50], Train Loss: 0.1555, Val Loss: 0.1587
Epoch [12/50], Train Loss: 0.1488, Val Loss: 0.1554
Epoch [13/50], Train Loss: 0.1425, Val Loss: 0.1548
Epoch [14/50], Train Loss: 0.1367, Val Loss: 0.1563
Epoch [15/50], Train Loss: 0.1314, Val Loss: 0.1594
Epoch [16/50], Train Loss: 0.1267, Val Loss: 0.1636
Epoch [17/50], Train Loss: 0.1226, Val Loss: 0.1683
Epoch [18/50], Train Loss: 0.1193, Val Loss: 0.1730
Epoch [19/50], Train Loss: 0.1165, Val Loss: 0.1770
Epoch [20/50], Train 

In [ ]:
# --------------------
# 7. Evaluate on Test Data
# --------------------
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_t)
    test_loss = criterion(test_outputs, y_test_t)
print(f"Final Test Loss (MSE): {test_loss.item():.4f}")

Final Test Loss (MSE): 0.3269


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# --------------------
# 7. Evaluate on Test Data
# --------------------
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_t)
    test_loss = criterion(test_outputs, y_test_t)
print(f"Final Test Loss (MSE): {test_loss.item():.4f}")

# Convert back to numpy
y_pred = test_outputs.numpy()
y_true = y_test_t.numpy()

# Inverse scaling to original units
y_pred = scaler_y.inverse_transform(y_pred)
y_true = scaler_y.inverse_transform(y_true)

# Metrics
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print(f"MSE (scikit-learn): {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R² Score: {r2:.4f}")

Final Test Loss (MSE): 0.3269
MSE (scikit-learn): 174.2265
RMSE: 13.1995
MAE: 7.8823
R² Score: 0.6532
